# board
**Depends on:** 02_food, HexMagic.primitives, HexMagic.climate

The hex board and spatial queries — no pieces or squads yet.

In [ ]:
#| default_exp magic/board

In [ ]:
#| export
from __future__ import annotations
from datetime import datetime
import numpy as np
from dataclasses import dataclass, field
from fastcore.basics import patch
import matplotlib.colors as mcolors

In [ ]:
#| export
from fasthtml.common import *
from fasthtml.jupyter import *
from fasthtml.common import *

In [ ]:
#| default_exp magic/board

In [ ]:
#| export
from HexMagic.geology import Geology, DrainageBasins, Watershed
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.overlay import Terrain, HexGrid, HexPosition
from HexMagic.overlay import CreamOverlay, RiverOverlay,OceanWaveOverlay
from HexMagic.climate import TerrainFactory
from HexMagic.game.flag import CountryFlag , PieceType, GameContext, DiagramGlyphs
from HexMagic.styles import StyleCSS,  SVGBuilder, SVGDef
from HexMagic.primitives import HexTouchMap, HexPosition, HexGrid, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.primitives import MapRect, MapPath,MapCord,  MapSize, HexRegion

In [ ]:
#| export
from HexMagic.magic.food import FoodYield, HexNumberOverlay

In [ ]:
showDemo = False

## GameParts

In [ ]:
#| export
class GameParts:

    def __init__(self, world:Geology = None):
        if world is None:
            
            world = TerrainFactory.create_world(
                bounds=MapRect(MapCord(0, 0), MapSize(300, 300)),
                preset='temperate',
                name='Maiden Lane',
                radius=15,
                lon_span=10.0,
                num_plates=8,
                subdivisions=3,
                ocean_fraction=0.3,
                oceanic_sides=['N'],
                terrain_age='young',
                formation_type='ridge',
                elevation_scale=1.5,
                erosion_age=0.1,
                num_lakes=0,
                seed=23,
                debug=True
            )

        self.load_world(world)
        

    def load_world(self, world:Terrain,radius: int = 20):
        self.terr = world.terrain
        
        self.grid: HexGrid = self.terr.hexGrid
        self.referenceIndex: int = self.grid.middle
        self.grid.adjustRadius(radius)
        rivers = self.terr.carve_to_ocean(num_lakes=0)
        self.builder: SVGBuilder = self.grid.builder
        self.basins: DrainageBasins = world.basins
        for i in range(len(self.grid.hexes)):
            self.grid.hexes[i].label = str(i)

        fy: FoodYield = FoodYield(self.terr, self.basins)
        
        self.yields: np.ndarray = fy.compute()
        self.fy: FoodYield = fy
        self.squads = []
        self.tp = None
        self.turn = 0


In [ ]:
#| export

@patch
def hp2i(self: GameParts, hexpos: HexPosition) -> int:
    return self.grid.hexposition_to_index(hexpos=hexpos, origin_index=self.referenceIndex)

@patch
def i2hp(self: GameParts, index: int) -> HexPosition:
   return self.grid.index_to_hexposition(index=index, origin_index=self.referenceIndex)
        

In [ ]:
#| export
@patch
def location(self:GameParts,location, facing = HexPosition.W):
    index = self.hp2i(location)
    retPlace=MapPlace(location,facing)
    retPlace.level = self.terr.elevationLevel(index)
    retPlace.harvest  = int(self.fy.tiers[index])
    retPlace.parent = self
    return retPlace

In [ ]:
#| export
@patch
def location_by_index(self:GameParts,index, facing = HexPosition.W):
    location = self.i2hp(index)
    return self.location(location,facing)

In [ ]:
#| export
@patch
def overlayContext(parts: GameParts, *, region=None, padding=1, radius=None,
                   corridors=None, extra_squads=None, extra_pieces=None):
    """Build a GameContext from a GameParts instance."""
    terrain = parts.terr
    if region is not None:
        terrain = terrain.zoom(region, padding=padding)

    grid = terrain.hexGrid
    if radius:
        grid.adjustRadius(radius)

    squads = list(getattr(parts, 'squads', []))
    if extra_squads:
        squads.extend(extra_squads)

    pieces = []
    for sq in squads:
        pieces.extend([p for p in sq.pieces if p.place is not None])
    if extra_pieces:
        pieces.extend(extra_pieces)

    return GameContext(
        terrain=terrain, grid=grid, builder=grid.builder,
        c2f=getattr(terrain, 'c2f', None),
        extras=dict(
            basins=parts.basins,
            coarse_basins=parts.basins,
            corridors=corridors or [],
            pieces=pieces,
            squads=squads,
        )
    )


In [ ]:
#| export
@patch
def find_starts(self: GameParts, n: int, ring_radius: int = 3, 
                min_distance: int = 6, balance_tolerance: float = 0.3,
                top_k: int = 40) -> list[int]:

    result = self.fy.find_oasis( n = n,  ring_radius = ring_radius,
               min_distance = min_distance, balance_tolerance = balance_tolerance,
               top_k = top_k) 
    
    
    return result


In [ ]:
myStuff = GameParts()

In [ ]:
myStuff.find_starts(3)

## MapPlace

In [ ]:
#| export
@dataclass
class MapPlace:
    location: HexPosition = HexPosition.origin()
    facing: HexPosition = HexPosition.W
    level: int = 0
    harvest: int = 0
    parent: GameParts | None = None
    tenure: int = 0  # turns spent at this location
    pantry: int = 0  # per-piece food storage


## Overlays

In [ ]:
#| export
def FogOverlay( fill="#ffffff", stroke="#cccccc", **kw) -> OverlaySpec:
    """White out hexes with no terrain data."""
    def render(ctx: OverlayContext) -> str:
        c2f = getattr(ctx, 'c2f', None)
        if not c2f: return ""
        mapped = {ni for indices in c2f.values() for ni in (indices if isinstance(indices, list) else [indices])}
        fog = StyleCSS("fog", fill=fill, stroke=stroke, stroke_width=1)
        ctx.builder.add_style(fog)
        for i in range(len(ctx.grid.hexes)):
            if i not in mapped:
                ctx.grid.hexes[i].style = fog
        return ""
    return OverlaySpec("fog", render, priority=6)


an example of orienting sight
```python
def LanternOverlay(**kw) -> OverlaySpec:
    """Render sight regions for placed pieces using their lantern style, plus facing bars."""
    def render(ctx) -> str:
        grid, builder = ctx.grid, ctx.builder
        N = len(grid.hexes)
        r = grid.radius
        parts = []

        for piece in ctx.pieces:
            if piece.place is None: continue
            builder.add_style(piece.lantern)

            # Sight region
            for idx in sorted(piece.sight().hexes):
                for fi in ctx.fine_indices(idx):
                    if 0 <= fi < N:
                        h = grid.hexes[fi]
                        parts.append(Hex(h.radius, h.center, piece.lantern, v=h.v).svg())

            # Facing bar at piece's location
            loc_idx = piece.place.parent.hp2i(piece.place.location)
            for fi in ctx.fine_indices(loc_idx):
                if 0 <= fi < N:
                    c = grid.hexes[fi].center
                    glyphs = DiagramGlyphs(piece.flag, size=r * 0.7)
                    glyphs.register_styles(builder)
                    dirs = HexPosition.directions()
                    facing_dir = dirs.index(piece.place.facing) if piece.place.facing in dirs else 0
                    parts.append(glyphs.facing_bar(c, facing_dir))

        return '\n'.join(parts)

    return OverlaySpec("lanterns", render, requires={'pieces'}, priority=62)
```

In [ ]:
!cat ../../HexMagic/plot/*.py

In [ ]:
ctx = myStuff.overlayContext()

In [ ]:
TerrainDisplay(
        CreamOverlay(stylized=True),
        RiverOverlay(max_width=4),
        FogOverlay(),
        #OasisOverlay(),
        HexNumberOverlay(),
        OceanWaveOverlay(
        num_waves=5,
        spacing=8,
        amplitude_start=3.0,
        amplitude_decay=0.6,
        wavelength=40,
        stroke_color="#4a7fb5",
        opacity_start=0.5,
        opacity_decay=0.7,
        stroke_width=1.2),
        ctx=ctx,
        debug = not showDemo
    )

In [ ]:
??TerrainDisplay

I am looking to make an FT for MapPlace. the idea would be a minimap around it maybe a radius uf 3. and a sightbar point where it is. The hexRadius can be very small.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Nice idea! Here's how I'd think about structuring it:

1. **Get the region**: Use `parent.grid.indices_in_range(index, 3)` to get the hex indices around the location, wrap in a `HexRegion`.
2. **Build a context**: Use `parent.overlayContext(region=that_region, radius=8)` (or similar small radius) to get a zoomed `GameContext`.
3. **Render the minimap**: Pass to `TerrainDisplay` with just `CreamOverlay` + maybe `RiverOverlay`.
4. **Facing indicator**: Add a custom overlay that calls `DiagramGlyphs.facing_bar` at the center hex.

A couple of questions before you start:

- Do you want the `MapPlace` hex to always be at the center of the minimap, or just within the cropped region?
- Should the facing bar use the piece's flag/style (like in `LanternOverlay`), or just a simple arrow?

The center I think is good
lets use a facing bar

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good. `MapPlace` doesn't currently have a `flag` or `PieceType` attached to it — do you want to:

1. **Pass a flag/color in** as an optional param to the `__ft__` method (e.g. `flag=CountryFlag.RED`)
2. **Use a default/neutral glyph** (e.g. a fixed color arrow regardless of ownership)

lets use a neutral/fixed color

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good, keeps it simple. Here's the plan for `MapPlace.__ft__`:

```python
def __ft__(self: MapPlace):
    ...
```

The tricky part is that `overlayContext` takes a `region` as a `HexRegion`, but you currently just have the center index. You'd do something like:

```python
index = self.parent.hp2i(self.location)
nearby = self.parent.grid.indices_in_range(index, 3)
region = HexRegion(set(nearby), self.parent.grid)
ctx = self.parent.overlayContext(region=region, radius=8)
```

Does that look right to you, or would you rather just pass the raw index range directly without the region zoom?

I guess this needs to be a function that returns an FT since for instance HexPlace doesn't know about GameParts. So maybe there should be a function on GameParts that returns the FT

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good point on separation of concerns. Something like:

```python
def place_ft(self: GameParts, place: MapPlace) -> FT:
    ...
```

Though actually — `MapPlace` *does* have `parent: GameParts | None`, so it could call back to it. But a method on `GameParts` is cleaner and more explicit.

Do you want it as a `@patch` on `GameParts`, or a standalone function that takes both `parts` and `place`?

an @patch function with an optional color for the facingBar. we might want to put info about the pantry and tenure below

this was example of showing place stas using (piece.place is a mapPlace)
```python
def node_labels(self:SquadFlowRenderer, piece, flow: SquadFlow, nid, center: MapCord) -> str:
    """Name + vertically stacked SVG pill badges below node."""
    ring_r   = self.node_r + 6
    post_net = flow.post_net(nid)
    pantry   = piece.place.pantry if piece.place else 0
    tenure   = piece.place.tenure if piece.place else 0
    life     = piece.food.lifespan
    turns_left = max(0, life - tenure) if life < 999 else -1
    turns_str  = "∞" if turns_left < 0 else f"{turns_left}t"

    net_color  = "#4CAF50" if post_net >= 0 else "#E53935"
    net_bg     = "#e8f5e9" if post_net >= 0 else "#ffebee"
    pant_color = self.flag.darkPrimary

    cx = center.x
    y  = center.y + ring_r + 18  # start below ring
    dy = 22                       # vertical spacing between badges

    def pill(label, fg, bg, y_pos, w=70, h=16, r=8):
        return (
            f'<rect x="{cx - w/2:.1f}" y="{y_pos - h/2 - 1:.1f}" '
            f'width="{w}" height="{h}" rx="{r}" fill="{bg}" opacity="0.85"/>'
            f'<text x="{cx:.1f}" y="{y_pos:.1f}" text-anchor="middle" '
            f'dominant-baseline="central" font-size="12" font-weight="700" fill="{fg}">'
            f'{label}</text>'
        )

    parts = [
        # Name — no pill, just bold text
        f'<text x="{cx:.1f}" y="{y:.1f}" text-anchor="middle" '
        f'dominant-baseline="central" font-size="13" font-weight="700" '
        f'fill="{self.flag.darkPrimary}">{piece.name}</text>',

        # Net food pill
        pill(f"{post_net:+.0f} 🍖", net_color, net_bg, y + dy),

        # Pantry pill
        pill(f"🧺 {pantry}", pant_color, "#f5f5f5", y + dy * 2),

        # Turns pill — only show if finite
        *([] if turns_left < 0 else
          [pill(f"⏳ {turns_str}", "#666", "#f5f5f5", y + dy * 3, w=55)]),
    ]

    return '\n'.join(parts)
```

and this was another example where it was dynamically loaded in

```python
@patch
def _render_node(self: SquadFlowDiagram, nid, data, flow, svg_pos, builder) -> str:
    r = SquadFlowRenderer(flag=self.squad.flag, node_r=22.5 * self.node_scale)
    center = svg_pos[nid]
    piece = data['piece']

    # Piece chess icon from flag
    svg_str, pat_def = self.squad.flag.piece_svg(
        piece.type, center,
        scale=self.node_scale, size='board',
        piece_id=f"flow_{nid}_{piece.type.value}")
    if pat_def is not None:
        builder.add_definition(pat_def)

    # Death marker if starving
    post_net = flow.post_net(nid)
    glyphs = DiagramGlyphs(self.squad.flag, size=22.5 * self.node_scale)
    glyphs.register_styles(builder)
    death = ""
    if post_net < 0 and not data['is_king']:
        ring_r = 22.5 * self.node_scale + 6
        death = glyphs.death_marker(
            MapCord(center.x - ring_r * 0.7, center.y - ring_r * 0.7),
            size=22.5 * self.node_scale * 0.4)

    hx_attrs = ""
    if self.on_click:
        hx_attrs = ' '.join(f'{k}="{v}"' for k, v in self.on_click(piece).items())

    parts = []
    parts = [f'<g style="cursor:pointer" {hx_attrs}>' if hx_attrs else '<g>',
             r.node_ring_fragment(nid, data, flow, center, builder),
             svg_str,
             death,
             #r.node_food_pip(nid, flow, center, builder),
             #r.node_labels(piece, flow, nid, center),
             '</g>']

    #parts = []

    label = (self.label_fn(piece, flow, nid, center) 
    if self.label_fn else r.node_labels(piece, flow, nid, center))
    parts.append(label)
    return '\n'.join(p for p in parts if p)
```

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's a first pass — let me write it out for you to review and tweak:

```python
@patch
def place_ft(self: GameParts, place: MapPlace, color: str = "#5c6bc0") -> FT:
    index = self.hp2i(place.location)
    nearby = set(self.grid.indices_in_range(index, 3))
    region = HexRegion(nearby, self.grid)
    ctx = self.overlayContext(region=region, radius=8)

    # Facing bar overlay
    dirs = HexPosition.directions()
    facing_dir = dirs.index(place.facing) if place.facing in dirs else 0
    glyphs = DiagramGlyphs(
        CountryFlag(primary=color, dark_primary=color),
        size=ctx.grid.radius * 0.7
    )
    glyphs.register_styles(ctx.builder)

    # Find center hex in ctx grid
    ctx_center = ctx.grid.hexposition_to_index(HexPosition.origin(), ctx.grid.middle)
    center = ctx.grid.hexes[ctx_center].center
    facing_svg = glyphs.facing_bar(center, facing_dir)
    ctx.builder.adjust("facing", facing_svg)
    ctx.grid.update()

    # Pill badges below
    def pill(label):
        return Span(label, style="background:#f0f0f0; border-radius:8px; padding:2px 8px; font-size:11px; margin:2px;")

    return Div(
        Div(NotStr(ctx.builder.xml())),
        Div(pill(f"🧺 {place.pantry}"), pill(f"⏳ {place.tenure}t"),
            style="display:flex; justify-content:center; flex-wrap:wrap;"),
    )
```

A few things to confirm before you run it:

1. Does `CountryFlag` accept `primary=` / `dark_primary=` as kwargs, or does it need a different constructor for a neutral flag?
2. Does `ctx.builder.adjust("facing", ...)` work as a new layer, or do you need to register it first via `ctx.builder.updateLayers`?

In [ ]:
#| export
@patch
def place_ft(self: GameParts, place: MapPlace, *overlays: OverlaySpec, ctx = None,
             distance: int = 3, radius: int = 8, color: str = "#5c6bc0") -> FT:
    index = self.hp2i(place.location)
    nearby = set(self.grid.indices_in_range(index, distance))
    region = HexRegion(nearby, self.grid)
    if ctx is None:
        ctx = self.overlayContext(region=region, radius=radius)

    # Run overlays — mutates ctx.builder as a side effect
    TerrainDisplay(*overlays, ctx=ctx)

    # Facing bar on top
    rgb = mcolors.to_rgb(color)
    flag = CountryFlag(rgb, "place", 1900)
    dirs = HexPosition.directions()
    facing_dir = dirs.index(place.facing) if place.facing in dirs else 0
    glyphs = DiagramGlyphs(flag, size=ctx.grid.radius * 0.7)
    glyphs.register_styles(ctx.builder)
    ctx_center = ctx.grid.hexposition_to_index(HexPosition.origin(), ctx.grid.middle)
    center = ctx.grid.hexes[ctx_center].center
    ctx.builder.adjust("facing", glyphs.facing_bar(center, facing_dir))

    def pill(label):
        return Span(label, style=(
            "background:#f0f0f0; border-radius:8px; padding:2px 8px;"
            " font-size:11px; margin:2px; font-family:sans-serif;"))

    harvest_color = ["#e57373","#ffb74d","#aed581","#66bb6a","#26a69a"][min(place.harvest, 4)]

    return Div(
        Div(NotStr(ctx.builder.xml()), style="line-height:0;"),
        Div(
            pill(f"🧺 {place.pantry}"),
            pill(f"⏳ {place.tenure}t"),
            Span(f"{'★' * (place.harvest+1)}",
                 style=f"font-size:13px; color:{harvest_color}; margin:2px;"),
            style="display:flex; justify-content:center; align-items:center; flex-wrap:wrap; padding:3px 0;"
        ),
        style="display:inline-block; border:1px solid #ddd; border-radius:6px; overflow:hidden; background:white;"
    )


In [ ]:
myStuff = GameParts()

In [ ]:
oasis = myStuff.fy.find_oasis(n=3)

In [ ]:
kingPlace = myStuff.location_by_index(oasis[0])

In [ ]:
myStuff.place_ft(kingPlace)

We need to add some overlays to this

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

It's rendering but all hexes are `sealevel` — the terrain colours aren't being applied. The likely culprits to add:

- `CreamOverlay(stylized=True)` — terrain colours
- `RiverOverlay` — rivers

Should I add those two to the `TerrainDisplay` call inside `place_ft`, or do you want to pass overlays in as a parameter?

there isn't yet a TerrainDisplay inside of def place_ft(self: GameParts, place: MapPlace, color: str = "#5c6bc0") -> FT:

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Right! Currently `place_ft` builds the context and calls `ctx.grid.update()` directly — it never runs any overlays. You need to either:

1. **Use `TerrainDisplay`** to render the minimap with overlays (replacing the raw `ctx.builder.xml()`)
2. **Manually apply** `CreamOverlay` to `ctx` before calling `ctx.grid.update()`

Which would you prefer?

I would like to be able to call

```python
myStuff.place_ft(kingPlace,
 CreamOverlay(stylized=True),
        RiverOverlay(max_width=4),
        FogOverlay(),
        #OasisOverlay(),
        HexNumberOverlay(),

)
```
The terrain display would be built, and its builder would do the xml()
below would be information about the pantry and yield

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

So `place_ft` needs to accept `*overlays` as extra args, then pass them to `TerrainDisplay` with `ctx=ctx` and `debug=True` to get the builder back rather than a `HexTouchMap`.

Looking at `TerrainDisplay`'s signature — when `debug=True` it returns `ctx.builder.__ft__()`. That gives you the debug view, not the raw SVG. 

Do you have a way to get just `ctx.builder.xml()` after `TerrainDisplay` has run its overlays, or would you need to inline the overlay-running logic from `TerrainDisplay` directly?

In [ ]:
oasis = myStuff.fy.find_oasis(n=3)

# make three places with different facings / stats
places = []
for i, (idx, facing, pantry, tenure) in enumerate([
    (oasis[0], HexPosition.NE, 5, 3),
    (oasis[1], HexPosition.W,  0, 0),
    (oasis[2], HexPosition.SE, 12, 7),
]):
    p = myStuff.location_by_index(idx, facing)
    p.pantry = pantry
    p.tenure = tenure
    places.append(p)

overlays = (
    CreamOverlay(stylized=True),
    RiverOverlay(max_width=4),
    FogOverlay(),
    HexNumberOverlay(),
)

Div(
    *[myStuff.place_ft(p, *overlays, distance=1,radius=20) for p in places],
    style="display:flex; gap:16px; flex-wrap:wrap; padding:12px;"
)
